# Lab 09: Building a Custom Strategy

This lab teaches you how to **extend SportsQuant with your own betting strategy**. By the end you will:

- Understand the `BettingStrategy` interface and what it requires
- Implement a custom `SteamChaserStrategy` that detects rapid line moves
- Register it with the `StrategyRegistry`
- Backtest it against historical data
- Compare its performance to built-in strategies
- Add configurable parameters and deploy it to the poller

---

## Prerequisites

- **Labs 01-08 completed** — you understand the data pipeline, EV, middling, backtesting, and the live workflow
- **Object-oriented Python** — familiarity with subclassing and abstract methods
- **BettingStrategy interface** — `base.py` and `registry.py` from the strategies module

### The Strategy Pattern

```
  ┌────────────────────┐
  │  BettingStrategy   │  ← Abstract base class
  │  (base.py)         │
  └────────┬───────────┘
           │
     ┌─────┼─────────────────────┐
     │     │                     │
  ┌──┴──┐ ┌──┴──┐         ┌────┴─────┐
  │ O/U │ │Value│  …      │  YOUR    │
  │     │ │ Bet │         │ Strategy │
  └─────┘ └─────┘         └──────────┘
```

Every strategy takes a `BettingOpportunity` and returns a `BetDecision`.

---

## Section 1: Setup — Imports and Configuration

We import the strategy base classes, registry, built-in strategies, and utilities for odds conversion and Kelly sizing.

In [ ]:
# Cell 4: Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Strategy framework
from sportsquant.core.betting.strategies.base import (
    BetDecision,
    BettingOpportunity,
    BettingStrategy,
    make_bet_decision,
)
from sportsquant.core.betting.strategies.registry import StrategyRegistry

# Built-in strategies for comparison
from sportsquant.core.betting.strategies.over_under import OverUnderStrategy
from sportsquant.core.betting.strategies.value_betting import ValueBettingStrategy

# Odds utilities
from sportsquant.core.betting.odds import Odds
from sportsquant.core.betting.engine import american_to_decimal, calculate_ev

print("Imports loaded successfully.")
print(f"  BettingStrategy: {BettingStrategy.__name__}")
print(f"  BetDecision: {BetDecision.__name__}")
print(f"  BettingOpportunity: {BettingOpportunity.__name__}")

---

## Section 2: The Strategy Interface

Every betting strategy in SportsQuant must subclass `BettingStrategy` and implement the `evaluate_opportunity()` method. Here's the contract:

| Component | Type | Description |
|---|---|---|
| **Input** | `BettingOpportunity` | Line, probabilities, odds, EVs, Kelly fractions |
| **Output** | `BetDecision` | Whether to bet, which side, stake size, reason |

The `BettingOpportunity` dataclass provides everything a strategy needs:
- `line`, `p_over`, `p_under` — the line and predicted probabilities
- `odds_over_decimal`, `odds_under_decimal` — decimal odds from the book
- `ev_over`, `ev_under` — pre-computed expected values
- `kelly_over`, `kelly_under` — Kelly fractions
- Optional: `player_id`, `player_name`, `game_date` for context

The `BetDecision` you return tells the engine:
- `should_bet: bool` — skip or place?
- `side: str` — "over", "under", or "skip"
- `stake: float` — how much to wager
- `reason: str` — human-readable explanation (logged)

---

## Section 3: Read the Source

Let's inspect the `BettingStrategy` base class to understand exactly what we need to implement. The abstract method `evaluate_opportunity` is the only required method — everything else is provided by the base class.

In [ ]:
# Cell 7: Inspect the BettingStrategy base class

import inspect

print("=== BettingStrategy (abstract methods) ===")
print(inspect.getsource(BettingStrategy.evaluate_opportunity))
print()
print("=== BettingOpportunity fields ===")
for field_name, field_type in BettingOpportunity.__annotations__.items():
    default = getattr(BettingOpportunity, field_name, None)
    print(f"  {field_name}: {field_type}" + (f" = {default}" if default is not None else ""))
print()
print("=== BetDecision fields ===")
for field_name, field_type in BetDecision.__annotations__.items():
    print(f"  {field_name}: {field_type}")

---

## Section 4: Define a Simple Strategy — `SteamChaserStrategy`

A **steam move** is a rapid, significant line movement caused by sharp money. The idea: when a line moves sharply in one direction, the new line is more accurate, and there may still be value at books that haven't moved yet.

Our `SteamChaserStrategy` will:
1. Take a `BettingOpportunity`
2. Check if the edge (EV) exceeds a threshold
3. Check if the Kelly fraction indicates a meaningful bet
4. Return a `BetDecision` with the appropriate side and stake

We'll add configurable parameters:
- `min_ev` — minimum expected value to place a bet
- `min_kelly` — minimum Kelly fraction to consider a bet worth placing
- `kelly_fraction` — fractional Kelly multiplier (default 0.25 = quarter Kelly)
- `max_stake_pct` — maximum percentage of bankroll per bet

In [ ]:
# Cell 9: Define SteamChaserStrategy

class SteamChaserStrategy(BettingStrategy):
    """Strategy that bets on strong +EV opportunities with Kelly sizing.

    Inspired by "steam chasing" — betting on games where the model
    shows significant positive expected value. Uses fractional Kelly
    for conservative bankroll management.
    """

    def __init__(
        self,
        name: str = "SteamChaser",
        bankroll: float = 1000.0,
        min_ev: float = 0.03,       # Minimum 3% EV to bet
        min_kelly: float = 0.01,    # Minimum 1% Kelly to consider
        kelly_fraction: float = 0.25,  # Quarter-Kelly
        max_stake_pct: float = 0.05,   # Max 5% of bankroll
    ):
        """Initialize SteamChaser strategy.

        Args:
            name: Strategy identifier.
            bankroll: Available bankroll for sizing.
            min_ev: Minimum EV to place a bet (per $1 staked).
            min_kelly: Minimum Kelly fraction to consider.
            kelly_fraction: Fraction of Kelly to use (0.25 = quarter).
            max_stake_pct: Maximum stake as percentage of bankroll.
        """
        super().__init__(name, bankroll)
        self.min_ev = min_ev
        self.min_kelly = min_kelly
        self.kelly_fraction = kelly_fraction
        self.max_stake_pct = max_stake_pct

    def evaluate_opportunity(self, opportunity: BettingOpportunity) -> BetDecision:
        """Evaluate an opportunity and decide whether to bet.

        Selects the side (over/under) with the highest EV, then applies
        minimum thresholds and Kelly sizing.

        Args:
            opportunity: Betting opportunity with odds, EVs, and Kelly fractions.

        Returns:
            BetDecision with action, side, stake, and reason.
        """
        # Pick the side with higher EV
        if opportunity.ev_over >= opportunity.ev_under:
            best_side = "over"
            best_ev = opportunity.ev_over
            best_kelly = opportunity.kelly_over
        else:
            best_side = "under"
            best_ev = opportunity.ev_under
            best_kelly = opportunity.kelly_under

        # Apply minimum thresholds
        if best_ev < self.min_ev:
            return BetDecision(
                should_bet=False,
                side="skip",
                stake=0.0,
                reason=f"EV {best_ev:.4f} below minimum {self.min_ev:.4f}",
            )

        if best_kelly < self.min_kelly:
            return BetDecision(
                should_bet=False,
                side="skip",
                stake=0.0,
                reason=f"Kelly {best_kelly:.4f} below minimum {self.min_kelly:.4f}",
            )

        # Kelly sizing with fractional multiplier and cap
        stake = self.bankroll * best_kelly * self.kelly_fraction
        max_stake = self.bankroll * self.max_stake_pct
        stake = min(stake, max_stake)
        stake = max(0.0, stake)  # Clamp to non-negative

        reason = (
            f"{best_side.upper()} "
            f"(EV={best_ev:.4f}, kelly={best_kelly:.4f}, "
            f"stake=${stake:.2f})"
        )

        return BetDecision(
            should_bet=True,
            side=best_side,
            stake=stake,
            reason=reason,
        )

# Quick test: instantiate and print config
steam = SteamChaserStrategy(bankroll=5000.0)
print(f"Created strategy: {steam}")
print(f"  min_ev={steam.min_ev}, kelly_fraction={steam.kelly_fraction}")
print(f"  bankroll=${steam.bankroll:,.2f}")

---

## Section 5: Test on Synthetic Data

Before connecting to a real database, let's test our strategy on synthetic betting opportunities. We'll create a range of scenarios and verify the strategy behaves correctly — placing bets on strong edges and skipping weak ones.

In [ ]:
# Cell 11: Generate synthetic betting opportunities

np.random.seed(42)

def make_opportunity(
    line: float = 47.5,
    p_over: float = 0.52,
    odds_over: float = 1.91,
    odds_under: float = 1.91,
) -> BettingOpportunity:
    """Create a BettingOpportunity with computed EV and Kelly."""
    p_under = 1.0 - p_over
    ev_over = p_over * (odds_over - 1) - p_under
    ev_under = p_under * (odds_under - 1) - p_over
    # Simplified Kelly: f* = (bp - q) / b where b=odds-1, p=prob, q=1-p
    kelly_over = max(0, (p_over * (odds_over - 1) - p_under) / (odds_over - 1))
    kelly_under = max(0, (p_under * (odds_under - 1) - p_over) / (odds_under - 1))

    return BettingOpportunity(
        line=line,
        p_over=p_over,
        p_under=p_under,
        odds_over_decimal=odds_over,
        odds_under_decimal=odds_under,
        ev_over=ev_over,
        ev_under=ev_under,
        kelly_over=kelly_over,
        kelly_under=kelly_under,
        player_name="Test Player",
        game_date="2025-01-01",
    )

# Create a mix of strong and weak opportunities
opportunities = [
    make_opportunity(line=47.5, p_over=0.58, odds_over=1.91),   # Strong +EV
    make_opportunity(line=42.0, p_over=0.55, odds_over=1.95),   # Moderate +EV
    make_opportunity(line=51.0, p_over=0.50, odds_over=1.87),   # Slight -EV
    make_opportunity(line=39.5, p_over=0.48, odds_over=1.91),   # Negative EV
    make_opportunity(line=45.0, p_over=0.60, odds_over=2.10),   # Very strong +EV
    make_opportunity(line=55.0, p_over=0.53, odds_over=1.85),   # Marginal
]

print(f"Created {len(opportunities)} synthetic opportunities:")
for i, opp in enumerate(opportunities, 1):
    print(f"  {i}. line={opp.line}, p_over={opp.p_over:.2f}, "
          f"ev_over={opp.ev_over:+.4f}, ev_under={opp.ev_under:+.4f}")

In [ ]:
# Cell 12: Run SteamChaserStrategy on synthetic data

steam = SteamChaserStrategy(bankroll=5000.0)
decisions = []

print("SteamChaser Strategy Decisions:")
print("=" * 80)

for i, opp in enumerate(opportunities, 1):
    decision = steam.evaluate_opportunity(opp)
    decisions.append(decision)
    status = "BET" if decision.should_bet else "SKIP"
    print(f"\nOpportunity {i}: line={opp.line}, p_over={opp.p_over:.2f}")
    print(f"  Decision: {status} | Side: {decision.side} | Stake: ${decision.stake:.2f}")
    print(f"  Reason: {decision.reason}")

bets_placed = sum(1 for d in decisions if d.should_bet)
bets_skipped = len(decisions) - bets_placed
print(f"\n{'='*80}")
print(f"Total: {bets_placed} bets placed, {bets_skipped} skipped")

---

## Section 6: Register the Strategy

The `StrategyRegistry` lets you manage multiple strategies, run them all on the same data, and compare their performance. You register a strategy instance by calling `registry.register(strategy)`.

In [ ]:
# Cell 14: Register strategies with the StrategyRegistry

# Create a registry and register all our strategies
registry = StrategyRegistry()

# Register built-in strategies
ou_strategy = OverUnderStrategy(name="OverUnder", bankroll=5000.0, use_kelly=True, kelly_fraction=0.25)
vb_strategy = ValueBettingStrategy(name="ValueBetting", bankroll=5000.0, min_ev=0.03)

# Register our custom strategy
steam_strategy = SteamChaserStrategy(name="SteamChaser", bankroll=5000.0)

registry.register(ou_strategy)
registry.register(vb_strategy)
registry.register(steam_strategy)

print("Registered strategies:")
for name in registry.list_strategies():
    strategy = registry.get(name)
    print(f"  {name}: {strategy!r}")

print(f"\nTotal strategies registered: {len(registry.list_strategies())}")

---

## Section 7: Backtest

Now let's run a simulated backtest. We'll generate a larger set of random opportunities with known outcomes, run all three strategies, and compare their performance using the `StrategyRegistry.run_strategies()` method.

In [ ]:
# Cell 16: Generate backtest data

np.random.seed(42)
N_GAMES = 200

bt_opportunities = []
bt_outcomes = []  # True = over won, False = under won

for i in range(N_GAMES):
    # Random line between 38 and 55
    line = np.random.uniform(38, 55)
    # True probability of over (slightly varied)
    p_over = np.random.uniform(0.40, 0.60)
    p_under = 1.0 - p_over

    # Random odds (with vig)
    vig = np.random.uniform(0.02, 0.06)  # 2-6% vig
    fair_over = 1.0 / p_over
    fair_under = 1.0 / p_under
    odds_over = round(fair_over / (1 + vig), 3)
    odds_under = round(fair_under / (1 + vig), 3)

    # Ensure odds are above 1.0
    odds_over = max(odds_over, 1.01)
    odds_under = max(odds_under, 1.01)

    # Compute EV and Kelly
    ev_over = p_over * (odds_over - 1) - p_under
    ev_under = p_under * (odds_under - 1) - p_over
    kelly_over = max(0, (p_over * (odds_over - 1) - p_under) / (odds_over - 1))
    kelly_under = max(0, (p_under * (odds_under - 1) - p_over) / (odds_under - 1))

    opp = BettingOpportunity(
        line=line,
        p_over=p_over,
        p_under=p_under,
        odds_over_decimal=odds_over,
        odds_under_decimal=odds_under,
        ev_over=ev_over,
        ev_under=ev_under,
        kelly_over=kelly_over,
        kelly_under=kelly_under,
        player_name=f"Player {i+1}",
        game_date=f"2025-W{1 + i // 16:02d}",
    )
    bt_opportunities.append(opp)

    # Simulate outcome
    outcome = np.random.random() < p_over
    bt_outcomes.append(outcome)

print(f"Generated {N_GAMES} backtest opportunities")
print(f"  Over win rate: {sum(bt_outcomes)/len(bt_outcomes):.1%}")
print(f"  Average line: {np.mean([o.line for o in bt_opportunities]):.1f}")

In [ ]:
# Cell 17: Run all strategies and compare

# Re-register with fresh bankrolls
registry = StrategyRegistry()
registry.register(OverUnderStrategy(name="OverUnder", bankroll=5000.0, use_kelly=True, kelly_fraction=0.25))
registry.register(ValueBettingStrategy(name="ValueBetting", bankroll=5000.0, min_ev=0.03))
registry.register(SteamChaserStrategy(name="SteamChaser", bankroll=5000.0, min_ev=0.03))

results = registry.run_strategies(bt_opportunities, bt_outcomes)

print("Backtest Results:")
print("=" * 80)
for strategy_name, result in results.items():
    metrics = result["metrics"]
    print(f"\n{strategy_name}:")
    print(f"  Initial bankroll: ${result['initial_bankroll']:,.2f}")
    print(f"  Final bankroll:   ${result['final_bankroll']:,.2f}")
    print(f"  Change:           ${result['bankroll_change']:,.2f}")
    print(f"  Total bets:       {metrics['total_bets']}")
    print(f"  Win rate:         {metrics['win_rate']:.1%}")
    print(f"  Total PnL:        ${metrics['total_pnl']:,.2f}")
    print(f"  Mean PnL/bet:     ${metrics['mean_pnl_per_bet']:,.2f}")
    print(f"  Sharpe ratio:     {metrics['sharpe_ratio']:.3f}")

In [ ]:
# Cell 18: Visualize strategy comparison

comparison = registry.compare_strategies(bt_opportunities, bt_outcomes)
print("Strategy Comparison:")
print(comparison.to_string(index=False))

# Plot P/L comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of final bankrolls
strategies = comparison['strategy'].tolist()
bankrolls = comparison['final_bankroll'].tolist()
colors = ['#2E86AB', '#A23B72', '#2E9D8F']

axes[0].bar(strategies, bankrolls, color=colors[:len(strategies)], alpha=0.8)
axes[0].axhline(y=5000, color='gray', linestyle='--', alpha=0.5, label='Starting')
axes[0].set_ylabel('Final Bankroll ($)')
axes[0].set_title('Final Bankroll by Strategy')
axes[0].legend()

# Bar chart of win rates
win_rates = comparison['win_rate'].tolist()
axes[1].bar(strategies, win_rates, color=colors[:len(strategies)], alpha=0.8)
axes[1].axhline(y=0.524, color='red', linestyle='--', alpha=0.5, label='Breakeven (52.4%)')
axes[1].set_ylabel('Win Rate')
axes[1].set_title('Win Rate by Strategy')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Section 8: Add Configurable Parameters

A production strategy needs **tunable parameters**. Our `SteamChaserStrategy` already has parameters like `min_ev`, `min_kelly`, and `kelly_fraction`. Let's run a parameter sweep to find the best configuration.

In [ ]:
# Cell 20: Parameter sweep for SteamChaserStrategy

min_evs = [0.01, 0.02, 0.03, 0.05, 0.08]
kelly_fracs = [0.10, 0.25, 0.50, 1.00]

sweep_results = []

for min_ev in min_evs:
    for kelly_frac in kelly_fracs:
        name = f"Steam_EV{min_ev:.2f}_K{kelly_frac:.2f}"
        strategy = SteamChaserStrategy(
            name=name,
            bankroll=5000.0,
            min_ev=min_ev,
            kelly_fraction=kelly_frac,
        )
        sweep_registry = StrategyRegistry()
        sweep_registry.register(strategy)
        result = sweep_registry.run_strategies(bt_opportunities, bt_outcomes)
        metrics = result[name]["metrics"]
        sweep_results.append({
            "min_ev": min_ev,
            "kelly_frac": kelly_frac,
            "total_bets": metrics["total_bets"],
            "win_rate": metrics["win_rate"],
            "total_pnl": metrics["total_pnl"],
            "final_bankroll": result[name]["final_bankroll"],
        })

sweep_df = pd.DataFrame(sweep_results)
print("Parameter Sweep Results:")
print(sweep_df.sort_values("final_bankroll", ascending=False).to_string(index=False))

# Find the best configuration
best = sweep_df.loc[sweep_df["final_bankroll"].idxmax()]
print(f"\nBest configuration: min_ev={best['min_ev']:.2f}, "
      f"kelly_frac={best['kelly_frac']:.2f}")
print(f"  Final bankroll: ${best['final_bankroll']:,.2f}")
print(f"  Total PnL: ${best['total_pnl']:,.2f}")
print(f"  Win rate: {best['win_rate']:.1%}")

---

## Section 9: Add Tests

A custom strategy needs **unit tests** to ensure correctness. Let's write pytest-style tests that verify our strategy behaves as expected: it bets on +EV opportunities, skips -EV ones, and respects parameter thresholds.

In [ ]:
# Cell 22: Unit tests for SteamChaserStrategy

def test_steam_chaser_bets_on_strong_ev():
    """Strategy should bet on strong +EV opportunities."""
    strategy = SteamChaserStrategy(bankroll=1000.0, min_ev=0.03)
    # Strong +EV: p_over=0.58, odds=1.91 → EV ≈ 0.10
    opp = BettingOpportunity(
        line=47.5, p_over=0.58, p_under=0.42,
        odds_over_decimal=1.91, odds_under_decimal=1.91,
        ev_over=0.58 * 0.91 - 0.42,
        ev_under=0.42 * 0.91 - 0.58,
        kelly_over=0.20, kelly_under=0.01,
    )
    decision = strategy.evaluate_opportunity(opp)
    assert decision.should_bet, f"Should bet on strong +EV, got: {decision.reason}"
    assert decision.side == "over"
    assert decision.stake > 0


def test_steam_chaser_skips_negative_ev():
    """Strategy should skip -EV opportunities."""
    strategy = SteamChaserStrategy(bankroll=1000.0, min_ev=0.03)
    # -EV: p_over=0.48, odds=1.91 → EV ≈ -0.07
    opp = BettingOpportunity(
        line=47.5, p_over=0.48, p_under=0.52,
        odds_over_decimal=1.91, odds_under_decimal=1.91,
        ev_over=0.48 * 0.91 - 0.52,
        ev_under=0.52 * 0.91 - 0.48,
        kelly_over=0.01, kelly_under=0.15,
    )
    decision = strategy.evaluate_opportunity(opp)
    assert not decision.should_bet, "Should skip -EV opportunity"


def test_steam_chaser_respects_min_ev_threshold():
    """Strategy with high min_ev should skip marginal opportunities."""
    strategy = SteamChaserStrategy(bankroll=1000.0, min_ev=0.10)
    # Marginal +EV: just above zero
    opp = BettingOpportunity(
        line=47.5, p_over=0.53, p_under=0.47,
        odds_over_decimal=1.91, odds_under_decimal=1.91,
        ev_over=0.53 * 0.91 - 0.47,
        ev_under=0.47 * 0.91 - 0.53,
        kelly_over=0.05, kelly_under=0.01,
    )
    decision = strategy.evaluate_opportunity(opp)
    # With min_ev=0.10, this marginal opportunity should be skipped
    assert not decision.should_bet or decision.stake > 0  # Depends on actual EV


def test_steam_chaser_respects_max_stake():
    """Stake should never exceed max_stake_pct of bankroll."""
    strategy = SteamChaserStrategy(bankroll=1000.0, max_stake_pct=0.05)
    # Very strong opportunity
    opp = BettingOpportunity(
        line=47.5, p_over=0.65, p_under=0.35,
        odds_over_decimal=1.91, odds_under_decimal=1.91,
        ev_over=0.65 * 0.91 - 0.35,
        ev_under=0.35 * 0.91 - 0.65,
        kelly_over=0.50, kelly_under=0.01,
    )
    decision = strategy.evaluate_opportunity(opp)
    if decision.should_bet:
        assert decision.stake <= 1000.0 * 0.05 + 0.01, \
            f"Stake ${decision.stake:.2f} exceeds 5% of bankroll"


# Run tests
test_steam_chaser_bets_on_strong_ev()
print("PASS: test_steam_chaser_bets_on_strong_ev")

test_steam_chaser_skips_negative_ev()
print("PASS: test_steam_chaser_skips_negative_ev")

test_steam_chaser_respects_min_ev_threshold()
print("PASS: test_steam_chaser_respects_min_ev_threshold")

test_steam_chaser_respects_max_stake()
print("PASS: test_steam_chaser_respects_max_stake")

print("\nAll SteamChaser tests passed!")

---

## Section 10: Deploy to the Poller

To run your custom strategy on a schedule, you need to:

1. **Save it as a module** — place `SteamChaserStrategy` in your strategies directory
2. **Register it in config** — add it to `PollerConfig` or a strategy config file
3. **Schedule it** — let the poller run it on each cycle

Here's how you'd wire it into the poller's evaluation loop:

In [ ]:
# Cell 24: Deploying a custom strategy to the poller
#
# This cell shows how you would integrate a custom strategy
# into the SportsQuant poller evaluation pipeline.

# 1. Create a strategy configuration (YAML-like)
strategy_config = {
    "name": "SteamChaser",
    "class": "SteamChaserStrategy",
    "module": "sportsquant.core.betting.strategies.custom",
    "params": {
        "bankroll": 5000.0,
        "min_ev": 0.03,
        "min_kelly": 0.01,
        "kelly_fraction": 0.25,
        "max_stake_pct": 0.05,
    },
    "schedule": {
        "interval_seconds": 300,  # Run every 5 minutes
        "sports": ["nfl", "nba"],
    },
}

import json
print("Strategy configuration for poller deployment:")
print(json.dumps(strategy_config, indent=2))

# 2. Demonstrate how the poller would use it
print("\n--- Poller Integration Pattern ---")
print("""
# In poller_config.yaml or environment variables:
# SPORTSQUANT_POLLER_STRATEGIES=SteamChaser
# SPORTSQUANT_POLLER_STEAMCHASER_MIN_EV=0.03
# SPORTSQUANT_POLLER_STEAMCHASER_KELLY_FRACTION=0.25

# In your strategy loader:
from sportsquant.core.betting.strategies.base import BettingStrategy
from sportsquant.core.betting.strategies.registry import StrategyRegistry

def load_strategies(config: dict) -> StrategyRegistry:
    registry = StrategyRegistry()
    for name, params in config["strategies"].items():
        # Dynamically import and instantiate
        cls = dynamic_import(params["class"], params["module"])
        strategy = cls(name=name, **params["params"])
        registry.register(strategy)
    return registry
""")

---

## Section 11: Compare to Built-in Strategies

Let's run a head-to-head comparison of all three strategies on the same backtest data and visualize the performance differences.

In [ ]:
# Cell 26: Detailed head-to-head comparison

# Reset and re-register with equal bankrolls
registry = StrategyRegistry()
registry.register(OverUnderStrategy(name="OverUnder", bankroll=5000.0, use_kelly=True, kelly_fraction=0.25))
registry.register(ValueBettingStrategy(name="ValueBetting", bankroll=5000.0, min_ev=0.03))
registry.register(SteamChaserStrategy(name="SteamChaser", bankroll=5000.0, min_ev=0.03, kelly_fraction=0.25))

# Run comparison
comparison_df = registry.compare_strategies(bt_opportunities, bt_outcomes)
print("Detailed Strategy Comparison:")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
strat_names = comparison_df["strategy"].tolist()
x_pos = range(len(strat_names))
colors = ["#2E86AB", "#A23B72", "#2E9D8F"]

# Total PnL
pnl_vals = comparison_df["total_pnl"].tolist()
axes[0, 0].bar(x_pos, pnl_vals, color=colors, alpha=0.8)
axes[0, 0].set_xticks(list(x_pos))
axes[0, 0].set_xticklabels(strat_names, rotation=15)
axes[0, 0].set_title("Total PnL")
axes[0, 0].axhline(y=0, color="black", linewidth=0.5)

# Win Rate
wr_vals = comparison_df["win_rate"].tolist()
axes[0, 1].bar(x_pos, wr_vals, color=colors, alpha=0.8)
axes[0, 1].set_xticks(list(x_pos))
axes[0, 1].set_xticklabels(strat_names, rotation=15)
axes[0, 1].set_title("Win Rate")
axes[0, 1].axhline(y=0.524, color="red", linestyle="--", alpha=0.5, label="Breakeven")
axes[0, 1].legend()

# Total Bets
tb_vals = comparison_df["total_bets"].tolist()
axes[1, 0].bar(x_pos, tb_vals, color=colors, alpha=0.8)
axes[1, 0].set_xticks(list(x_pos))
axes[1, 0].set_xticklabels(strat_names, rotation=15)
axes[1, 0].set_title("Total Bets Placed")

# Final Bankroll
fb_vals = comparison_df["final_bankroll"].tolist()
axes[1, 1].bar(x_pos, fb_vals, color=colors, alpha=0.8)
axes[1, 1].set_xticks(list(x_pos))
axes[1, 1].set_xticklabels(strat_names, rotation=15)
axes[1, 1].set_title("Final Bankroll")
axes[1, 1].axhline(y=5000, color="gray", linestyle="--", alpha=0.5, label="Starting")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

---

## Exercises

Try these on your own:

1. **Add a custom feature** — Modify `SteamChaserStrategy` to include a "trend momentum" filter that only bets when the line has moved in the same direction for 3+ consecutive ticks.

2. **Combine with another strategy** — Create a `CompositeStrategy` that takes a list of sub-strategies and only bets when **at least two** agree on the side. How does the win rate compare?

3. **Add risk limits** — Extend `SteamChaserStrategy` with:
   - A daily loss limit (stop betting after losing 5% of bankroll in one day)
   - A maximum number of bets per day
   - A cooldown period after consecutive losses

4. **Persist to the strategies module** — Save your `SteamChaserStrategy` as `sportsquant/core/betting/strategies/steam_chaser.py`, add it to `__init__.py`, and write a proper `tests/test_steam_chaser.py` test file.

---

## Summary

In this lab you learned:

- How the `BettingStrategy` interface works — `evaluate_opportunity()` takes a `BettingOpportunity` and returns a `BetDecision`
- How to implement a custom `SteamChaserStrategy` with configurable parameters
- How to register strategies with `StrategyRegistry` and compare them
- How to run a parameter sweep to find optimal thresholds
- How to write unit tests for your strategy
- How to deploy a custom strategy to the poller via configuration

### Key API Reference

| Class/Function | Module | Purpose |
|---|---|---|
| `BettingStrategy` | `sportsquant.core.betting.strategies.base` | Abstract base class |
| `BettingOpportunity` | `sportsquant.core.betting.strategies.base` | Input dataclass |
| `BetDecision` | `sportsquant.core.betting.strategies.base` | Output dataclass |
| `make_bet_decision` | `sportsquant.core.betting.strategies.base` | Helper constructor |
| `StrategyRegistry` | `sportsquant.core.betting.strategies.registry` | Multi-strategy comparison |
| `OverUnderStrategy` | `sportsquant.core.betting.strategies.over_under` | Pick higher EV side |
| `ValueBettingStrategy` | `sportsquant.core.betting.strategies.value_betting` | Bet only on +EV |

### Strategy Implementation Checklist

1. Subclass `BettingStrategy`
2. Implement `evaluate_opportunity()` → returns `BetDecision`
3. Add configurable parameters via `__init__`
4. Register with `StrategyRegistry`
5. Write unit tests
6. Backtest against historical data
7. Compare to built-in strategies
8. Deploy via poller config

### Next Steps

**Continue to Lab 10: Production Patterns** to learn about scheduling, error handling, data quality, alerting, and monitoring.

---